In [0]:
from pyspark.sql.functions import when, count, avg, sum,col

In [0]:
df_silver = spark.table("project_cardio.silver.cardio_clean")

In [0]:

df_age_group = (
    df_silver
    .withColumn(
        "age_group",
        when(col("age_years") < 30, "<30")
        .when((col("age_years") >= 30) & (col("age_years") < 40), "30-39")
        .when((col("age_years") >= 40) & (col("age_years") < 50), "40-49")
        .when((col("age_years") >= 50) & (col("age_years") < 60), "50-59")
        .otherwise("60+")
    )
    .groupBy("age_group")
    .agg(
        count("*").alias("nb_patients"),
        avg("bmi").alias("avg_bmi"),
        avg("tension_systolic").alias("avg_tension_systolic"),
        sum("cardio").alias("nb_cardio_risk")
    )
)

df_age_group.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("project_cardio.gold.cardio_age_group_metrics")